# 计算 LTO 分数

根据 `embed_fu.ipynb` 得到的《明一统志》各府文本向量和 `embed_LTOitem.ipynb` 中 bert-ancient-chinese 得到的 LTO 文言句句子的向量，计算各府LTO分数。

## 导入模块与路径

In [ ]:
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import pandas as pd

# 府文本和题项文本向量的路径
EMB_DIR = Path("../data/fu_embeddings")
LTO_DIR = Path("../data/LTOitem_embeddings/lto_ancient_embeddings.npy")

SENT_CSV = Path("../data/LTO_sentences.csv")
OUT_DIR = Path("../data/LTO_fu_scores.csv")


## 计算LTO分数
1. 计算每个府所有窗口与所有 LTO 句子嵌入的余弦相似度矩阵，形状为 `(窗口数, LTO句子数)`。
2. 将 LTO 句子按题项模块分组（`m13`、`m14`、`m19`、`m22`），对每个窗口分别计算 4 个模块的平均相似度，得到 `mod_scores`，形状为 `(窗口数, 4)`。
3. 将所有府的窗口分数纵向拼接为 `all_scores`，形状为 `总窗口数 × 4`，分别计算四个模块的全局标准差，将每个模块分数转换为 -0.5 * std + 3，模拟调查问卷的1-5计分，高分代表不同意
4. 将映射后的分数按府取平均，得到每个府的 4 个模块得分，形状为 `府数 × 4`。
5. 按照公式 `40 * (m13 - m14) + 25 * (m19 - m22) `计算最后的分数。

In [ ]:
# 府嵌入文件列表
npy_files = []
for root, dirs, files in os.walk(EMB_DIR):
    for fn in sorted(files):
        if fn.endswith(".npy"):
            npy_files.append(Path(root) / fn)
print(f"Embedding files: {len(npy_files)}")

lto_sentences = pd.read_csv(SENT_CSV, encoding="utf-8-sig")
lto_sentences = lto_sentences[lto_sentences["is_guwen"] == True].reset_index(drop=True)
lto_embeddings = np.load(LTO_DIR)

# 计算相似度
all_window_scores = []
all_fu_data = {}

for i, npy_path in enumerate(npy_files):
    rel = npy_path.relative_to(EMB_DIR)
    # rel is like: 江西布政使司/南昌府.npy
    prov = rel.parent.name
    fu_name = rel.stem
    windows = np.load(npy_path)
    sim = cosine_similarity(windows, lto_embeddings)
    mod_scores = np.zeros((windows.shape[0], 4))
    for j, mod in enumerate(["m13", "m14", "m19", "m22"]):
        mod_scores[:, j] = sim[:, lto_sentences["模块"] == mod].mean(axis=1)
    all_fu_data[(prov, fu_name)] = mod_scores
    all_window_scores.append(mod_scores)
    if (i + 1) % 50 == 0 or (i + 1) == len(npy_files):
        print(f"  [{i + 1}/{len(npy_files)}]")

# 把相似度反向处理，然后标准化缩放，模拟调查问卷的1-5计分，5非常不同意
all_scores = np.concatenate(all_window_scores, axis=0)
mean = all_scores.mean(axis=0, keepdims=True)
std = all_scores.std(axis=0, keepdims=True)

z = (all_scores - mean) / std
mapped = - 0.5 * z + 3

# 计算最终LTO分数，减去最小值平移到[0, 100]
offset = 0
fu_results = []
for (prov, fu_name), mod_scores in all_fu_data.items():
    n = mod_scores.shape[0]
    fu_mean = mapped[offset:offset + n].mean(axis=0)
    offset += n
    m13, m14, m19, m22 = fu_mean
    lto_raw = 40 * (m13 - m14) + 25 * (m19 - m22)
    fu_results.append({
        "NAME_CH": fu_name,
        "province": prov,
        "m13_交友": m13,
        "m14_节俭": m14,
        "m19_国家自豪": m19,
        "m22_坚持": m22,
        "LTO_raw": lto_raw,
        "windows": n,
    })

fu_results = pd.DataFrame(fu_results)
fu_results["LTO"] = fu_results["LTO_raw"] - fu_results["LTO_raw"].min()
print(f"Constant: {fu_results['LTO_raw'].min()}")
print(f"LTO Min: {fu_results['LTO'].min()}, Max: {fu_results['LTO'].max()}")

fu_results.to_csv(OUT_DIR, index=False)
print(f"Saved results to {OUT_DIR}")